In [ ]:
# Ajustar el directorio de trabajo al de la raíz del proyecto si ejecutamos desde la carpeta notebooks
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Directorio de trabajo actual:", os.getcwd())


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Configuración de estilo para los gráficos (estilo académico)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

In [30]:
def haversine(lat1, lon1, lat2, lon2):
    """Calcula la distancia en línea recta (km) entre dos coordenadas geográficas"""
    R = 6371.0 # Radio de la Tierra en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def run_eda():
    print("Iniciando el Análisis Exploratorio de Datos (EDA)...")
    
    # =====================================================================
    # 1. CARGA DE DATOS
    # =====================================================================
    try:
        print("Cargando datasets...")
        viajes = pd.read_csv('resultados/viajes.csv', low_memory=False)
        etapas = pd.read_csv('resultados/etapas.csv', low_memory=False)
        tarjetas = pd.read_csv('resultados/tarjetas.csv', low_memory=False)
        indices_h3 = pd.read_csv('data/indices_h3.csv', low_memory=False)
        paradas = pd.read_csv('data/paradas.csv', low_memory=False)
        tarifas = pd.read_csv('data/tarifas.csv', low_memory=False)
    except FileNotFoundError as e:
        print(f"Error al cargar los archivos: {e}")
        print("Asegurate de descargar los .csv del repositorio y mantener la estructura de carpetas.")
        return

    # =====================================================================
    # 2. CALIDAD Y LIMPIEZA DE DATOS
    # =====================================================================
    print("\n--- Estadísticas Generales ---")
    total_etapas = len(etapas)
    
    print(f"Total de etapas válidas: {total_etapas:,}")
    print(f"Total de tarjetas únicas: {len(tarjetas):,}")
    print(f"Total de viajes completos: {len(viajes):,}")
    
    if 'parada_id_d' in etapas.columns:
        etapas_validas = etapas.dropna(subset=['parada_id_d'])
        tasa_exito = (len(etapas_validas) / total_etapas) * 100
        print(f"Tasa de éxito en imputación de destinos (etapas): {tasa_exito:.1f}%")

    # =====================================================================
    # 3. CRUCE ESPACIAL Y CÁLCULO DE DISTANCIAS
    # =====================================================================
    print("\nProcesando tipologías geográficas y distancias con datos reales...")
    
    dict_parada_h3 = dict(zip(paradas['id'], paradas['h3']))
    dict_parada_lat = dict(zip(paradas['id'], paradas['latitud']))
    dict_parada_lon = dict(zip(paradas['id'], paradas['longitud']))
    
    viajes['h3_o'] = viajes['parada_id_o'].map(dict_parada_h3)
    viajes['h3_d'] = viajes['parada_id_d'].map(dict_parada_h3)
    
    viajes['lat_o'] = viajes['parada_id_o'].map(dict_parada_lat)
    viajes['lon_o'] = viajes['parada_id_o'].map(dict_parada_lon)
    viajes['lat_d'] = viajes['parada_id_d'].map(dict_parada_lat)
    viajes['lon_d'] = viajes['parada_id_d'].map(dict_parada_lon)

    mask_caba = indices_h3['provincia'].str.contains('CABA|Ciudad Autónoma|Capital Federal', case=False, na=False)
    indices_h3['jurisdiccion'] = np.where(mask_caba, 'CABA', 'PBA')
    
    dict_jurisdiccion = dict(zip(indices_h3['h3'], indices_h3['jurisdiccion']))
    
    viajes['jur_origen'] = viajes['h3_o'].map(dict_jurisdiccion)
    viajes['jur_destino'] = viajes['h3_d'].map(dict_jurisdiccion)
    
    def clasificar_viaje(row):
        if row['jur_origen'] == 'CABA' and row['jur_destino'] == 'CABA':
            return 'Intra-CABA'
        elif row['jur_origen'] == 'PBA' and row['jur_destino'] == 'PBA':
            return 'Intra-PBA'
        elif pd.notna(row['jur_origen']) and pd.notna(row['jur_destino']):
            return 'CABA-PBA'
        else:
            return 'Desconocido'
            
    viajes['tipo_viaje'] = viajes.apply(clasificar_viaje, axis=1)
    
    viajes_geo = viajes[viajes['tipo_viaje'] != 'Desconocido'].copy()
    viajes_geo['distancia'] = haversine(viajes_geo['lat_o'], viajes_geo['lon_o'], viajes_geo['lat_d'], viajes_geo['lon_d'])

    viajes_geo['etapas_agrupadas'] = viajes_geo['cantidad_etapas'].apply(
        lambda x: '1 etapa' if x == 1 else ('2 etapas' if x == 2 else '3+ etapas')
    )

    # ---------------------------------------------------------------------
    # 3.4 Identificación del Modo de Transporte
    # ---------------------------------------------------------------------
    print("Clasificando combinaciones de modos de transporte...")
    for col in ['etapas_colectivo', 'etapas_tren', 'etapas_subte']:
        if col in viajes_geo.columns:
            viajes_geo[col] = viajes_geo[col].fillna(0)
        else:
            viajes_geo[col] = 0

    c_col = viajes_geo['etapas_colectivo'] > 0
    c_tren = viajes_geo['etapas_tren'] > 0
    c_sub = viajes_geo['etapas_subte'] > 0

    num_modos = c_col.astype(int) + c_tren.astype(int) + c_sub.astype(int)

    condiciones = [
        (num_modos == 1) & c_col,
        (num_modos == 1) & c_tren,
        (num_modos == 1) & c_sub,
        (num_modos > 1) & c_col & c_tren & ~c_sub,
        (num_modos > 1) & c_col & c_sub & ~c_tren,
        (num_modos > 1) & c_tren & c_sub & ~c_col,
        (num_modos == 3)
    ]
    elecciones = [
        'Solo Colectivo', 'Solo Tren', 'Solo Subte',
        'Col + Tren', 'Col + Subte', 'Tren + Subte',
        'Col + Tren + Subte'
    ]
    viajes_geo['modo_transporte'] = np.select(condiciones, elecciones, default='Desconocido/Otros')

    # ---------------------------------------------------------------------
    # 3.5 Análisis de Tarifas y Descuentos
    # ---------------------------------------------------------------------
    print("Cruzando datos de tarifas (usando id_tarjeta + id_viaje)...")
    
    if 'id_tarifa' in etapas.columns and 'id_tarjeta' in etapas.columns:
        etapas['id_tarjeta'] = etapas['id_tarjeta'].astype(str).str.replace(r'\.0$', '', regex=True)
        etapas['id_viaje'] = etapas['id_viaje'].astype(str).str.replace(r'\.0$', '', regex=True)
        viajes_geo['id_tarjeta'] = viajes_geo['id_tarjeta'].astype(str).str.replace(r'\.0$', '', regex=True)
        viajes_geo['id_viaje'] = viajes_geo['id_viaje'].astype(str).str.replace(r'\.0$', '', regex=True)
        
        etapas['id_tarifa'] = pd.to_numeric(etapas['id_tarifa'], errors='coerce')
        tarifas['id'] = pd.to_numeric(tarifas['id'], errors='coerce')

        tarifas_por_viaje = etapas.dropna(subset=['id_tarifa']).drop_duplicates(subset=['id_tarjeta', 'id_viaje'], keep='first')[['id_tarjeta', 'id_viaje', 'id_tarifa']]
        
        viajes_geo = viajes_geo.merge(tarifas_por_viaje, on=['id_tarjeta', 'id_viaje'], how='left')
        
        tarifas_merge = tarifas[['id', 'nombre']].rename(columns={'id': 'id_tarifa', 'nombre': 'nombre_tarifa'})
        
        viajes_geo = viajes_geo.merge(tarifas_merge, on='id_tarifa', how='left')
        viajes_geo['nombre_tarifa'] = viajes_geo['nombre_tarifa'].fillna('Desconocida')
        
        es_plena_o_desc = viajes_geo['nombre_tarifa'].str.upper().isin(['TARIFA PLENA', 'DESCONOCIDA'])
        viajes_geo['perfil_tarifario'] = np.where(es_plena_o_desc, 'Tarifa Plena', 'Tarifa Social / Subsidio')
        
    else:
        print("Aviso: Faltan columnas clave en etapas.csv.")
        viajes_geo['perfil_tarifario'] = 'Desconocido'
        viajes_geo['nombre_tarifa'] = 'Desconocida'

    # =====================================================================
    # 4. FIGURA 1: Intermodalidad por Geografía
    # =====================================================================
    print("\nGenerando Figura 1...")
    plt.figure(figsize=(10, 6))
    
    prop_df = (viajes_geo.groupby('tipo_viaje')['etapas_agrupadas']
               .value_counts(normalize=True)
               .rename('porcentaje')
               .reset_index())
    prop_df['porcentaje'] *= 100

    # IMPRESIÓN DE TABLA:
    print("\n--- DATOS FIGURA 1 (Porcentaje de viajes por etapas) ---")
    tabla_fig1 = prop_df.pivot(index='tipo_viaje', columns='etapas_agrupadas', values='porcentaje').round(1)
    print(tabla_fig1)
    print("--------------------------------------------------------")

    ax1 = sns.barplot(data=prop_df, x='tipo_viaje', y='porcentaje', hue='etapas_agrupadas', 
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'], 
                      hue_order=['1 etapa', '2 etapas', '3+ etapas'])
    
    for container in ax1.containers:
        ax1.bar_label(container, fmt='%.1f%%', padding=3, size=10)
    
    plt.title('Figura 1: Porcentaje de viajes según cantidad de etapas por zona', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Cantidad de Etapas')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura1_intermodalidad.png', dpi=300)
    plt.close()

    # =====================================================================
    # 5. FIGURA 2: Distancias y Outliers por Geografía
    # =====================================================================
    print("\nGenerando Figura 2...")
    plt.figure(figsize=(10, 6))
    
    viajes_plot = viajes_geo[(viajes_geo['distancia'] > 0) & (viajes_geo['distancia'] < 100)]
    
    # IMPRESIÓN DE TABLA:
    print("\n--- DATOS FIGURA 2 (Estadísticas de distancia en km) ---")
    stats_dist = viajes_plot.groupby('tipo_viaje')['distancia'].describe()[['count', 'mean', '50%', 'max']].round(1)
    stats_dist.rename(columns={'count':'Cantidad', 'mean':'Promedio', '50%':'Mediana', 'max':'Máximo'}, inplace=True)
    print(stats_dist)
    print("----------------------------------------------------------")

    ax2 = sns.boxplot(data=viajes_plot, x='tipo_viaje', y='distancia', 
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'],
                      hue='tipo_viaje', legend=False,
                      palette='Set2', showfliers=True, flierprops={'marker':'o', 'markersize':3, 'alpha':0.3})
    
    medians = viajes_plot.groupby('tipo_viaje')['distancia'].median()
    orden = ['Intra-CABA', 'Intra-PBA', 'CABA-PBA']
    for i, categoria in enumerate(orden):
        if categoria in medians:
            valor_mediana = medians[categoria]
            ax2.text(i, valor_mediana + 0.5, f' {valor_mediana:.1f} km', 
                     horizontalalignment='center', size=11, color='black', weight='bold',
                     bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
    
    plt.title('Figura 2: Distribución de distancias de viaje según zona geográfica', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Distancia (km)')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura2_distancias.png', dpi=300)
    plt.close()

    # =====================================================================
    # 6. FIGURA 3: Modo de Transporte
    # =====================================================================
    print("\nGenerando Figura 3...")
    plt.figure(figsize=(12, 7))
    
    prop_modo = (viajes_geo.groupby('tipo_viaje')['modo_transporte']
                 .value_counts(normalize=True)
                 .rename('porcentaje')
                 .reset_index())
    prop_modo['porcentaje'] *= 100

    # IMPRESIÓN DE TABLA:
    print("\n--- DATOS FIGURA 3 (Modos de transporte % por zona) ---")
    tabla_fig3 = prop_modo.pivot(index='tipo_viaje', columns='modo_transporte', values='porcentaje').fillna(0).round(1)
    print(tabla_fig3)
    print("-------------------------------------------------------")

    orden_modos = ['Solo Colectivo', 'Col + Tren', 'Solo Tren', 'Col + Subte', 'Solo Subte', 'Col + Tren + Subte', 'Tren + Subte']
    orden_modos_presentes = [m for m in orden_modos if m in viajes_geo['modo_transporte'].unique()]

    ax3 = sns.barplot(data=prop_modo, x='tipo_viaje', y='porcentaje', hue='modo_transporte',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'],
                      hue_order=orden_modos_presentes, palette='tab10')
    
    for container in ax3.containers:
        labels = [f'{v.get_height():.1f}%' if v.get_height() > 0.5 else '' for v in container]
        ax3.bar_label(container, labels=labels, padding=3, size=9)
    
    plt.title('Figura 3: Composición Modal de Viajes por Jurisdicción', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Modo/Combinación', bbox_to_anchor=(1.05, 1), loc='upper left') 
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura3_modos.png', dpi=300)
    plt.close()

    # =====================================================================
    # 7. FIGURA 4: Perfil Tarifario por Zona
    # =====================================================================
    print("\nGenerando Figura 4 (Tarifas agrupadas)...")
    plt.figure(figsize=(10, 6))
    
    # Filtramos las desconocidas para el gráfico
    viajes_con_tarifa = viajes_geo[viajes_geo['nombre_tarifa'] != 'Desconocida'].copy()
    
    prop_tarifa = (viajes_con_tarifa.groupby('tipo_viaje')['perfil_tarifario']
                 .value_counts(normalize=True)
                 .rename('porcentaje')
                 .reset_index())
    prop_tarifa['porcentaje'] *= 100

    # IMPRESIÓN DE TABLA:
    print("\n--- DATOS FIGURA 4 (Perfil Tarifario % por zona) ---")
    tabla_fig4 = prop_tarifa.pivot(index='tipo_viaje', columns='perfil_tarifario', values='porcentaje').fillna(0).round(1)
    print(tabla_fig4)
    print("----------------------------------------------------------")

    ax4 = sns.barplot(data=prop_tarifa, x='tipo_viaje', y='porcentaje', hue='perfil_tarifario',
                      order=['Intra-CABA', 'Intra-PBA', 'CABA-PBA'], palette='Set1')
    
    for container in ax4.containers:
        ax4.bar_label(container, fmt='%.1f%%', padding=3, size=11, weight='bold')
    
    plt.title('Figura 4: Distribución de Tarifa Social vs Plena por Jurisdicción', fontweight='bold')
    plt.xlabel('Tipo de Viaje')
    plt.ylabel('Porcentaje (%)')
    plt.legend(title='Perfil Tarifario')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura4_tarifas.png', dpi=300)
    plt.close()

    # =====================================================================
    # 8. FIGURA 5: Desglose Específico de Viajes y Etapas por Tarifa
    # =====================================================================
    print("\nGenerando Figura 5 (Desglose exacto de tarifas)...")
    
    viajes_por_tarifa = viajes_con_tarifa.groupby('nombre_tarifa').size().reset_index(name='Viajes')

    etapas_merge = etapas.dropna(subset=['id_tarifa']).copy()
    etapas_merge['id_tarifa'] = pd.to_numeric(etapas_merge['id_tarifa'], errors='coerce')
    etapas_merge = etapas_merge.merge(tarifas_merge, on='id_tarifa', how='left')
    etapas_merge['nombre_tarifa'] = etapas_merge['nombre_tarifa'].fillna('Desconocida')
    
    etapas_por_tarifa = etapas_merge[etapas_merge['nombre_tarifa'] != 'Desconocida'].groupby('nombre_tarifa').size().reset_index(name='Etapas_Totales')

    resumen_tarifas = viajes_por_tarifa.merge(etapas_por_tarifa, on='nombre_tarifa', how='outer').fillna(0)
    
    resumen_tarifas['Prom_Etapas_por_Viaje'] = np.where(resumen_tarifas['Viajes'] > 0, 
                                                        (resumen_tarifas['Etapas_Totales'] / resumen_tarifas['Viajes']), 
                                                        0).round(2)
    
    resumen_tarifas = resumen_tarifas.sort_values(by='Etapas_Totales', ascending=False)
    
    # IMPRESIÓN DE TABLA:
    print("\n--- DATOS FIGURA 5 (Viajes y Etapas por Tarifa Específica) ---")
    print(resumen_tarifas.to_string(index=False, formatters={
        'Viajes': '{:,.0f}'.format,
        'Etapas_Totales': '{:,.0f}'.format
    }))
    print("--------------------------------------------------------------")

    # Gráfico
    plt.figure(figsize=(12, 8))
    top_tarifas = resumen_tarifas.head(15) 
    
    ax5 = sns.barplot(data=top_tarifas, x='Etapas_Totales', y='nombre_tarifa', hue='nombre_tarifa', palette='viridis', legend=False)
    for container in ax5.containers:
        ax5.bar_label(container, fmt='{:,.0f}', padding=5, size=10)

    ax5.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
    
    plt.title('Figura 5: Volumen de Etapas según Tipo de Tarifa Específica', fontweight='bold')
    plt.xlabel('Cantidad de Etapas Totales')
    plt.ylabel('Tipo de Tarifa')
    plt.tight_layout()
    plt.savefig('resultados/visualizaciones/figura5_detalle_tarifas.png', dpi=300)
    plt.close()
    print("\nEDA finalizado con éxito. Las 5 imágenes fueron guardadas en el directorio actual.")

    
    # 0. Preparar datos de tiempo (si no están en viajes_geo)
    print("Preparando datos de tiempo de viaje...")
    
    # SOLUCIÓN 1: Forzar claves a string limpias (sin decimales fantasma)
    # Eliminamos el ".0" si existiera al convertir flotantes a string
    etapas['id_tarjeta'] = etapas['id_tarjeta'].astype(str).str.replace(r'\.0$', '', regex=True)
    etapas['id_viaje'] = etapas['id_viaje'].astype(str).str.replace(r'\.0$', '', regex=True)
    viajes_geo['id_tarjeta'] = viajes_geo['id_tarjeta'].astype(str).str.replace(r'\.0$', '', regex=True)
    viajes_geo['id_viaje'] = viajes_geo['id_viaje'].astype(str).str.replace(r'\.0$', '', regex=True)
    
    if 'hora' in etapas.columns and 'hora_inicio' not in viajes_geo.columns:
        # NUEVA LÓGICA: La hora viene como un entero (ej: 15, 17, 7, 19)
        etapas['hora_decimal'] = pd.to_numeric(etapas['hora'], errors='coerce')
        
        tiempo_viaje = etapas.groupby(['id_tarjeta', 'id_viaje']).agg(
            hora_inicio=('hora_decimal', 'min'),
            hora_fin=('hora_decimal', 'max')
        ).reset_index()
        
        # Calcular duración en horas enteras
        tiempo_viaje['duracion_horas'] = tiempo_viaje['hora_fin'] - tiempo_viaje['hora_inicio']
        
        # Evitar negativos (viajes nocturnos que cruzan la medianoche)
        tiempo_viaje['duracion_horas'] = np.where(tiempo_viaje['duracion_horas'] < 0, tiempo_viaje['duracion_horas'] + 24, tiempo_viaje['duracion_horas'])
        
        # Rellenar NaNs con 0 por seguridad
        tiempo_viaje['duracion_horas'] = tiempo_viaje['duracion_horas'].fillna(0)
        
        # SOLUCIÓN 2 (La más importante): Como la hora no tiene minutos, viajes que empiezan
        # y terminan dentro de la misma hora (ej 15:10 a 15:45) tendrán duracion = 0.
        # Para evitar división por cero, a los viajes con duración "0" les asignamos un 
        # promedio estimado de 30 minutos (0.5 horas). Si tardó 1 hora bloque, se deja en 1.0.
        tiempo_viaje['duracion_horas'] = np.where(tiempo_viaje['duracion_horas'] == 0, 0.5, tiempo_viaje['duracion_horas'])
        
        viajes_geo = viajes_geo.merge(tiempo_viaje, on=['id_tarjeta', 'id_viaje'], how='left')
    
    
    print("Calculando Métricas de Fricción (Espacial y Temporal)...")
    
    # 1. Filtramos viajes válidos (distancia > 500m)
    # Ahora sí, duracion_horas será siempre >= 0.5
    viajes_validos = viajes_geo[(viajes_geo['distancia'] >= 0.5) & (viajes_geo['duracion_horas'].notna())].copy()
    
    # 2. Aplicamos fórmulas de Fricción
    # IFI (Espacial): Etapas por cada 10 KM
    viajes_validos['IFI'] = (viajes_validos['cantidad_etapas'] / viajes_validos['distancia']) * 10
    
    # IFT (Temporal): Etapas por Hora de viaje. 
    # Representa la "densidad de transbordos" en el tiempo.
    viajes_validos['IFT'] = viajes_validos['cantidad_etapas'] / viajes_validos['duracion_horas']
    
    # Velocidad Comercial Estimada (km/h) (Línea recta, sirve como proxy de ineficiencia)
    viajes_validos['Velocidad_Euclidiana'] = viajes_validos['distancia'] / viajes_validos['duracion_horas']
    
    # 3. Resultados agrupados por Tipo de Viaje
    print("\n--- MÉTRICAS DE FRICCIÓN MEDIANAS POR ZONA GEOGRÁFICA ---")
    print("IFI: Etapas por 10km | IFT: Etapas por Hora | Vel: Km/h en línea recta")
    resumen_zona = viajes_validos.groupby('tipo_viaje').agg(
        Mediana_IFI=('IFI', 'median'),
        Mediana_IFT=('IFT', 'median'),
        Mediana_Velocidad=('Velocidad_Euclidiana', 'median'),
        Duracion_Promedio_Minutos=('duracion_horas', lambda x: x.mean() * 60)
    ).reset_index()
    print(resumen_zona.round(2).to_string(index=False))
    
    # 4. Resultados agrupados por Perfil Tarifario
    print("\n--- ANÁLISIS DE PENALIZACIÓN POR TARIFA ESPECÍFICA (Top 10 volúmenes) ---")
    # Filtramos solo tarifas conocidas
    resumen_tarifa = viajes_validos[viajes_validos['nombre_tarifa'] != 'Desconocida'].groupby('nombre_tarifa').agg(
        Viajes=('id_viaje', 'count'),
        Promedio_Etapas=('cantidad_etapas', 'mean'),
        Duracion_Media_Minutos=('duracion_horas', lambda x: x.mean() * 60), 
        Mediana_IFI=('IFI', 'median'),
        Mediana_IFT=('IFT', 'median')
    ).reset_index()
    
    # Ordenamos por Promedio de Etapas de mayor a menor para ver la fricción
    resumen_tarifa = resumen_tarifa.sort_values(by='Promedio_Etapas', ascending=False).head(15)
    print(resumen_tarifa[['nombre_tarifa', 'Promedio_Etapas', 'Duracion_Media_Minutos', 'Mediana_IFI', 'Mediana_IFT']].round(2).to_string(index=False))

    # Asumiendo que tenés cargado el dataframe 'viajes_geo' en memoria
    # Filtramos primero los outliers (distancias en cero o mayores a 100km por errores de GPS)
    viajes_validos = viajes_geo[(viajes_geo['distancia'] > 0) & (viajes_geo['distancia'] < 100)]
    
    # 1. Cálculo Global
    media_global = viajes_validos['distancia'].mean()
    mediana_global = viajes_validos['distancia'].median()
    
    print("\n--- DISTANCIA GLOBAL DEL AMBA ---")
    print(f"La distancia MEDIA global es:   {media_global:.2f} km")
    print(f"La distancia MEDIANA global es: {mediana_global:.2f} km")
    print("---------------------------------")
    
    # 2. Cálculo por Jurisdicción (tipo_viaje)
    print("\n--- DISTANCIA POR JURISDICCIÓN ---")
    # Agrupamos por tipo de viaje y calculamos media, mediana y la cantidad total de viajes
    resumen_distancias = viajes_validos.groupby('tipo_viaje')['distancia'].agg(
        Media_km='mean',
        Mediana_km='median',
        Cantidad_Viajes='count'
    ).reset_index()
    
    # Redondeamos las distancias a 2 decimales
    resumen_distancias['Media_km'] = resumen_distancias['Media_km'].round(2)
    resumen_distancias['Mediana_km'] = resumen_distancias['Mediana_km'].round(2)
    
    # Imprimimos la tabla formateada
    print(resumen_distancias.to_string(index=False, formatters={
        'Cantidad_Viajes': '{:,.0f}'.format
    }))

In [31]:
if __name__ == "__main__":
    run_eda()

Iniciando el Análisis Exploratorio de Datos (EDA)...
Cargando datasets...

--- Estadísticas Generales ---
Total de etapas válidas: 9,173,945
Total de tarjetas únicas: 3,022,985
Total de viajes completos: 6,509,176
Tasa de éxito en imputación de destinos (etapas): 92.1%

Procesando tipologías geográficas y distancias con datos reales...
Clasificando combinaciones de modos de transporte...
Cruzando datos de tarifas (usando id_tarjeta + id_viaje)...

Generando Figura 1...

--- DATOS FIGURA 1 (Porcentaje de viajes por etapas) ---
etapas_agrupadas  1 etapa  2 etapas  3+ etapas
tipo_viaje                                    
CABA-PBA             38.1      46.7       15.3
Intra-CABA           80.2      18.2        1.6
Intra-PBA            72.6      23.2        4.1
--------------------------------------------------------

Generando Figura 2...

--- DATOS FIGURA 2 (Estadísticas de distancia en km) ---
             Cantidad  Promedio  Mediana  Máximo
tipo_viaje                                    